In [1]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

Device: cpu


In [4]:
import sys
from pathlib import Path

# Find the project root by searching upward for the src folder
current = Path.cwd()
PROJECT_ROOT = None

for path in [current] + list(current.parents):
    if (path / "src").is_dir():
        PROJECT_ROOT = path
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find the project root containing 'src'. "
        f"Current working directory: {current}"
    )

# Add project root to Python import path
sys.path.insert(0, str(PROJECT_ROOT))

print("Current working directory:", current)
print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").is_dir())

Current working directory: d:\Historical Image Restoration\notebooks
Project root: d:\Historical Image Restoration
src exists: True


In [5]:
from src.models import RestorationUNet

In [6]:
from src.models import RestorationUNet

model = RestorationUNet(
    in_channels=3,
    out_channels=3,
    base_channels=32,
    num_levels=4,
    residual_blocks_per_level=2,
)

model = model.to(DEVICE)

print("Model ready.")

Model ready.


In [7]:
from src.models import L1ReconstructionLoss

criterion = L1ReconstructionLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4
)

In [8]:
from datasets import load_dataset
from torch.utils.data import DataLoader

from src.datasets.openphoto_dataset import OpenPhotoRestoreDataset

dataset = load_dataset("joshuachin/openphoto-restore-dataset")

train_dataset = OpenPhotoRestoreDataset(
    dataset["train"],
    crop_size=256,
    training=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
)

In [9]:
import time

model.train()

start_time = time.time()

running_loss = 0.0

for batch_idx, batch in enumerate(train_loader):

    damaged = batch["damaged"].to(DEVICE)
    clean = batch["clean"].to(DEVICE)

    optimizer.zero_grad()

    prediction = model(damaged)

    loss = criterion(prediction, clean)

    loss.backward()

    optimizer.step()

    running_loss += loss.item()

    if (batch_idx + 1) % 10 == 0:
        print(
            f"Batch {batch_idx + 1}/100 "
            f"| Loss: {loss.item():.6f}"
        )

    if batch_idx + 1 >= 100:
        break

elapsed = time.time() - start_time

print("\n100-batch experiment complete.")
print("Average loss:", running_loss / 100)
print(f"Time: {elapsed:.2f} seconds")

Batch 10/100 | Loss: 0.316968
Batch 20/100 | Loss: 0.193466
Batch 30/100 | Loss: 0.246330
Batch 40/100 | Loss: 0.145745
Batch 50/100 | Loss: 0.119727
Batch 60/100 | Loss: 0.126801
Batch 70/100 | Loss: 0.116248
Batch 80/100 | Loss: 0.124024
Batch 90/100 | Loss: 0.144323
Batch 100/100 | Loss: 0.119918

100-batch experiment complete.
Average loss: 0.17131697393953801
Time: 897.97 seconds
